In [ ]:
!pip install transformers accelerate torch

\Load the Model into GPU Memory (Updated)

In [ ]:
import torch
from transformers import pipeline

print("⏳ Loading a fully un-gated open-source model into Colab memory...")

# Initializing Qwen 2.5 1.5B Instruct - fits beautifully and instantly on a T4 GPU
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Model loaded successfully on the GPU with zero authentication required!")

Define the Evaluation Function

In [ ]:
import json
import re

def interview_turn_local(job_title, job_desc, conversation_history, user_answer=None):
    """
    If user_answer is None, it's the start of the interview; generate question 1.
    Otherwise, evaluate user_answer and generate the next question.
    """

    # Precise instructions forcing the model to return both feedback AND the next question in JSON
    system_instruction = (
        "You are an expert cybersecurity and technical interviewer conducting a live interactive interview.\n"
        "Analyze the candidate's response against the job title and description.\n"
        "You must respond ONLY with a valid JSON object matching this exact schema:\n"
        "{\n"
        '  "score_content": "1-10 string or N/A if first question",\n'
        '  "score_delivery": "1-10 string or N/A if first question",\n'
        '  "feedback_strengths": "Brief statement of what they did well, or N/A if first question",\n'
        '  "feedback_weaknesses": "Brief constructive criticism, or N/A if first question",\n'
        '  "next_question": "Your next highly relevant technical or behavioral interview question"\n'
        "}"
    )

    # Set up the context prompt
    context_prompt = f"--- JOB CONTEXT ---\nJob Title: {job_title}\nKey Skills: {job_desc}\n\n"

    if user_answer is None:
        # Initial call to get the very first question
        user_content = f"{context_prompt}Start the interview. Generate the first technical or behavioral question."
    else:
        # Subsequent calls: pass the history context, the last question, and the user's answer
        user_content = f"{context_prompt}"
        if conversation_history:
            user_content += f"Previous Question Asked: {conversation_history[-1]['question']}\n"
        user_content += f"Candidate Answer: {user_answer}\n\nEvaluate this answer and provide the 'next_question'."

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    outputs = pipe(
        messages,
        max_new_tokens=600,
        do_sample=True,
        temperature=0.2, # Lower temperature enforces strict adherence to JSON layout
        top_p=0.9
    )

    raw_text = outputs[0]["generated_text"][-1]["content"].strip()

    try:
        # Extract json text specifically if the model added markdown blocks or extra characters
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return json.loads(raw_text)
    except Exception:
        # Graceful fallback text if parsing hiccups occur
        return {
            "score_content": "Completed",
            "score_delivery": "Completed",
            "feedback_strengths": "Answer recorded.",
            "feedback_weaknesses": "Could not parse precise metrics natively.",
            "next_question": "Can you elaborate on your experience with core security frameworks or protocols?"
        }

In [ ]:
def run_dynamic_interview():
    print("="*60)
    print("🤖 ACTIVE AI INTERVIEW COACH (5-QUESTION CHALLENGE)")
    print("="*60)

    job_title = input("\nEnter Target Job Title (e.g., Security Analyst): ")
    job_desc = input("Paste Job Description / Key Skills (e.g., nmap, wireshark): ")

    print("\n⏳ Initializing the interview engine and generating Question #1...")

    conversation_history = []
    MAX_QUESTIONS = 5

    # Get the very first question
    turn_data = interview_turn_local(job_title, job_desc, conversation_history, user_answer=None)
    current_question = turn_data.get("next_question", "Tell me about your experience in this field?")

    question_count = 1

    while question_count <= MAX_QUESTIONS:
        print("\n" + "="*50)
        print(f"🤖 INTERVIEWER QUESTION #{question_count} of {MAX_QUESTIONS}:")
        print(f"👉 {current_question}")
        print("="*50)

        user_answer = input("\n🗣️ Your Answer (or type 'exit' to quit): ")
        if user_answer.strip().lower() == 'exit':
            print("\n👋 Interview session terminated early. Good luck practicing!")
            return

        print("\n⏳ Interviewer is evaluating your response...")

        conversation_history.append({"question": current_question})

        # Get evaluation of the answer and fetch the next question
        turn_data = interview_turn_local(job_title, job_desc, conversation_history, user_answer=user_answer)

        # Display the validation dashboard for the answer just given
        print("\n📊 --- ANSWER #{count} VALIDATION ---".format(count=question_count))
        print(f"🔹 Content Accuracy : {turn_data.get('score_content')}/10")
        print(f"🔹 Delivery Metric  : {turn_data.get('score_delivery')}/10")
        print(f"✅ What went well   : {turn_data.get('feedback_strengths')}")
        print(f"⚠️ Where to improve : {turn_data.get('feedback_weaknesses')}")

        # Check if we just completed the final question
        if question_count == MAX_QUESTIONS:
            print("\n" + "="*60)
            print("🎉 INTERVIEW COMPLETE! You have answered all 5 questions.")
            print("🏆 Review your final feedback cards above to refine your prep.")
            print("="*60)
            break

        # Advance to the next question if we haven't hit the limit yet
        current_question = turn_data.get("next_question", "Can you walk me through your standard approach to troubleshooting?")
        question_count += 1

In [ ]:
if __name__ == "__main__":
    run_dynamic_interview()